### Attempt 2

In [ ]:
#!/usr/bin/env python3
"""
Isolate equilibrated gel slab and fit simulation box snugly around it.

Prepares gel for deformation tests (uniaxial compression, shear, etc.)

This script:
- Rotates mobile group (types 1,2,3) in x–y plane to align slab with axes
- Removes support (type 4) and piston (type 5) atoms
- Determines gel extent from polymer beads with clearance on ALL 6 faces
- Removes solvent/polymer outside the new box bounds
- Sets simulation box to fit snugly around gel
- Writes a new LAMMPS data file (no wall atoms)

Atom types in output: 1=polymer, 2=crosslinker, 3=solvent
"""

import numpy as np
from scipy.spatial import ConvexHull
import argparse


# =========================
# Group-consistent settings
# =========================
POLYMER_TYPES = {1, 2}        # group polymer
SOLVENT_TYPES = {3}           # group solvent
SUPPORT_TYPES = {4}           # support (removed)
PISTON_TYPES  = {5}           # piston (removed)

ROTATE_TYPES  = POLYMER_TYPES | SOLVENT_TYPES   # group mobile
BOX_CLEARANCE = 0.2                             # gap between gel and box faces


# =========================
# Geometry utilities
# =========================
def find_min_bounding_rect_angle(hull_points):
    """
    Find the rotation angle that gives the minimum area bounding rectangle.
    Uses rotating calipers on convex hull.
    """
    n = len(hull_points)
    min_area = float('inf')
    best_angle = 0
    
    for i in range(n):
        edge = hull_points[(i + 1) % n] - hull_points[i]
        edge_angle = np.arctan2(edge[1], edge[0])
        
        c, s = np.cos(-edge_angle), np.sin(-edge_angle)
        rotated = np.column_stack([
            c * hull_points[:, 0] - s * hull_points[:, 1],
            s * hull_points[:, 0] + c * hull_points[:, 1]
        ])
        
        width = rotated[:, 0].max() - rotated[:, 0].min()
        height = rotated[:, 1].max() - rotated[:, 1].min()
        area = width * height
        
        if area < min_area:
            min_area = area
            best_angle = edge_angle
    
    return best_angle


def rotate_mobile_atoms(atoms):
    """
    Rotate mobile group (types 1,2,3) to align minimum bounding rectangle with axes.
    """
    poly = [a for a in atoms if a['type'] in POLYMER_TYPES]
    if not poly:
        raise ValueError("No polymer atoms found")

    xy = np.array([[a['x'], a['y']] for a in poly])
    com = xy.mean(axis=0)

    hull = ConvexHull(xy)
    hull_points = xy[hull.vertices]
    angle = find_min_bounding_rect_angle(hull_points)

    print(f"Slab rotation angle (MABR): {np.degrees(angle):.2f} degrees")

    c, s = np.cos(-angle), np.sin(-angle)

    for a in atoms:
        if a['type'] in ROTATE_TYPES:
            x = a['x'] - com[0]
            y = a['y'] - com[1]
            a['x'] = c*x - s*y + com[0]
            a['y'] = s*x + c*y + com[1]

    return atoms


def find_gel_extent(atoms, clearance, percentile=0.1):
    """
    Gel extent from polymer beads with clearance on ALL 6 faces.
    Uses percentiles to exclude outliers.
    """
    poly = [a for a in atoms if a['type'] in POLYMER_TYPES]
    
    xs = np.array([a['x'] for a in poly])
    ys = np.array([a['y'] for a in poly])
    zs = np.array([a['z'] for a in poly])

    # Use percentiles to exclude outliers
    xmin = np.percentile(xs, percentile)
    xmax = np.percentile(xs, 100 - percentile)
    ymin = np.percentile(ys, percentile)
    ymax = np.percentile(ys, 100 - percentile)
    zmin = np.percentile(zs, percentile)
    zmax = np.percentile(zs, 100 - percentile)

    print(f"Polymer x range: {xmin:.2f} to {xmax:.2f} (width: {xmax-xmin:.2f})")
    print(f"Polymer y range: {ymin:.2f} to {ymax:.2f} (width: {ymax-ymin:.2f})")
    print(f"Polymer z range: {zmin:.2f} to {zmax:.2f} (height: {zmax-zmin:.2f})")

    return {
        'xmin': xmin - clearance,
        'xmax': xmax + clearance,
        'ymin': ymin - clearance,
        'ymax': ymax + clearance,
        'zmin': zmin - clearance,
        'zmax': zmax + clearance
    }


def remove_non_gel_atoms(atoms, ext):
    """
    Remove support, piston, and any atoms outside box bounds.
    Keeps only polymer (1,2) and solvent (3) inside the gel extent.
    """
    kept = []
    removed_support_piston = 0
    removed_outside = 0

    for a in atoms:
        # Remove support and piston entirely
        if a['type'] in SUPPORT_TYPES | PISTON_TYPES:
            removed_support_piston += 1
            continue
        
        # Remove atoms outside box bounds (all 6 faces)
        if (a['x'] < ext['xmin'] or a['x'] > ext['xmax'] or
            a['y'] < ext['ymin'] or a['y'] > ext['ymax'] or
            a['z'] < ext['zmin'] or a['z'] > ext['zmax']):
            removed_outside += 1
            continue
        
        kept.append(a)

    print(f"Removed {removed_support_piston} support/piston atoms")
    print(f"Removed {removed_outside} atoms outside box bounds")
    return kept


# =========================
# LAMMPS I/O
# =========================
def parse_lammps_data(filename):
    """Parse LAMMPS data file and extract all information."""
    
    atoms = []
    bonds = []
    box_bounds = {}
    masses = {}
    
    with open(filename, 'r') as f:
        lines = f.readlines()
    
    i = 0
    while i < len(lines):
        line = lines[i].strip()
        
        if 'xlo xhi' in line:
            parts = line.split()
            box_bounds['xlo'] = float(parts[0])
            box_bounds['xhi'] = float(parts[1])
        elif 'ylo yhi' in line:
            parts = line.split()
            box_bounds['ylo'] = float(parts[0])
            box_bounds['yhi'] = float(parts[1])
        elif 'zlo zhi' in line:
            parts = line.split()
            box_bounds['zlo'] = float(parts[0])
            box_bounds['zhi'] = float(parts[1])
        
        elif line == 'Masses':
            i += 2
            while i < len(lines) and lines[i].strip() and not lines[i].strip().startswith(('Atoms', 'Bonds', 'Pair')):
                parts = lines[i].split()
                if len(parts) >= 2:
                    try:
                        masses[int(parts[0])] = float(parts[1])
                    except ValueError:
                        pass
                i += 1
            continue
        
        elif line == 'Atoms' or line.startswith('Atoms'):
            i += 2
            while i < len(lines) and lines[i].strip() and not lines[i].strip().startswith(('Bonds', 'Velocities', 'Pair')):
                parts = lines[i].split()
                if len(parts) >= 6:
                    try:
                        atoms.append({
                            'id': int(parts[0]),
                            'mol': int(parts[1]),
                            'type': int(parts[2]),
                            'x': float(parts[3]),
                            'y': float(parts[4]),
                            'z': float(parts[5])
                        })
                    except ValueError:
                        pass
                i += 1
            continue
        
        elif line == 'Bonds' or line.startswith('Bonds'):
            i += 2
            while i < len(lines) and lines[i].strip():
                parts = lines[i].split()
                if len(parts) >= 4:
                    try:
                        bonds.append({
                            'id': int(parts[0]),
                            'type': int(parts[1]),
                            'atom1': int(parts[2]),
                            'atom2': int(parts[3])
                        })
                    except ValueError:
                        pass
                i += 1
            continue
        
        i += 1
    
    return atoms, bonds, box_bounds, masses


def write_lammps_data(filename, atoms, bonds, box, masses):
    """Write LAMMPS data file with updated box bounds."""
    old2new = {a['id']: i+1 for i, a in enumerate(atoms)}

    valid_bonds = [
        {'id': i+1, 'type': b['type'],
         'atom1': old2new[b['atom1']], 'atom2': old2new[b['atom2']]}
        for i, b in enumerate(bonds)
        if b['atom1'] in old2new and b['atom2'] in old2new
    ]

    # Count atom types present
    types_present = set(a['type'] for a in atoms)
    n_atom_types = max(types_present) if types_present else 3

    with open(filename, 'w') as f:
        f.write("LAMMPS data file - isolated gel for deformation\n\n")
        f.write(f"{len(atoms)} atoms\n")
        f.write(f"{len(valid_bonds)} bonds\n\n")
        f.write(f"{n_atom_types} atom types\n")
        f.write("1 bond types\n\n")
        f.write(f"{box['xlo']:.6f} {box['xhi']:.6f} xlo xhi\n")
        f.write(f"{box['ylo']:.6f} {box['yhi']:.6f} ylo yhi\n")
        f.write(f"{box['zlo']:.6f} {box['zhi']:.6f} zlo zhi\n\n")

        f.write("Masses\n\n")
        for i in range(1, n_atom_types + 1):
            f.write(f"{i} {masses.get(i, 1.0)}\n")

        f.write("\nAtoms\n\n")
        for i, a in enumerate(atoms, 1):
            f.write(f"{i} {a['mol']} {a['type']} "
                    f"{a['x']:.6f} {a['y']:.6f} {a['z']:.6f}\n")

        if valid_bonds:
            f.write("\nBonds\n\n")
            for b in valid_bonds:
                f.write(f"{b['id']} {b['type']} {b['atom1']} {b['atom2']}\n")


# =========================
# Main driver
# =========================
def isolate_gel(input_file, output_file, clearance=BOX_CLEARANCE, percentile=0.1):
    """
    Isolate gel from equilibrated slab and fit box snugly around it.
    
    Parameters:
        input_file: Path to equilibrated .data file
        output_file: Path for output .data file
        clearance: Gap between gel and box faces (default 0.2)
        percentile: Percentile for excluding outliers (default 0.1)
    """
    print("="*50)
    print(f"Isolating gel from: {input_file}")
    print(f"Box clearance: {clearance}")
    print("="*50)
    
    atoms, bonds, box, masses = parse_lammps_data(input_file)
    print(f"Read {len(atoms)} atoms, {len(bonds)} bonds")
    
    # Rotate to align with axes
    atoms = rotate_mobile_atoms(atoms)
    
    # Find gel extent with clearance on all faces
    ext = find_gel_extent(atoms, clearance, percentile)
    
    # Remove support, piston, and atoms outside bounds
    atoms = remove_non_gel_atoms(atoms, ext)
    
    # Set new box bounds to gel extent
    new_box = {
        'xlo': ext['xmin'],
        'xhi': ext['xmax'],
        'ylo': ext['ymin'],
        'yhi': ext['ymax'],
        'zlo': ext['zmin'],
        'zhi': ext['zmax']
    }
    
    print(f"\nNew box dimensions:")
    print(f"  Lx = {new_box['xhi'] - new_box['xlo']:.2f}")
    print(f"  Ly = {new_box['yhi'] - new_box['ylo']:.2f}")
    print(f"  Lz = {new_box['zhi'] - new_box['zlo']:.2f}")
    
    write_lammps_data(output_file, atoms, bonds, new_box, masses)
    
    print(f"\nWrote {len(atoms)} atoms to {output_file}")
    print("="*50)



In [ ]:
# Inputs

input_file = "../../lammps_data/slab_with_support/final_config_slab_support_5beads_tall_rho04_p1.5_1.0_1.0_600000.data"
output_file = "../../lammps_data/slab_with_support/isolated_slab_support_5beads_tall_rho04_p1.5_1.0_1.0_600000.data"
clearance = 0.2 # gap between gel and box faces (default is 0.2)
percentile = 0.1 # percentile for excluding outliers (default is 0.1)

isolate_gel(input_file, output_file, clearance, percentile)

